In [1]:
import json
import glob
import pandas as pd

In [31]:
# Gallery
artworks_dict = json.load(open("exports/production/artworks.json", "r"))
categories_dict = json.load(open("exports/production/categories.json", "r"))
techniques_dict = json.load(open("exports/production/techniques.json", "r"))

# Studies
studies_dict = json.load(open("exports/production/studies.json", "r"))
image_attachments_dict = json.load(open("exports/production/image_attachments.json", "r"))
attachments_dict = json.load(open("exports/production/attachments.json", "r"))
images_dict = json.load(open("exports/production/images.json", "r"))

# Intro slideshows
intro_slideshows_dict = json.load(open("exports/production/intro_slideshows.json", "r"))
slides_dict = json.load(open("exports/production/slides.json", "r"))


In [32]:
pd.DataFrame(artworks_dict).head()

,id,catno,title,date_created,xinch,yinch,xcm,ycm,recent,filename,created_at,updated_at,category_id,technique_id
0,1474,1,Victory,1/1/2002,24.0,16.0,60.0,40.0,0,1.jpg,0000-00-00 00:00:00,0000-00-00 00:00:00,5,5
1,1475,2,Dove 3,1/1/2002,16.0,28.0,40.0,70.0,0,2.jpg,0000-00-00 00:00:00,0000-00-00 00:00:00,5,5
2,1476,3,Flying 2,1/1/2002,16.0,40.0,40.0,100.0,0,3.jpg,0000-00-00 00:00:00,0000-00-00 00:00:00,5,5
3,1477,4,Dove 1,1/1/2002,20.0,40.0,50.0,100.0,0,4.jpg,0000-00-00 00:00:00,0000-00-00 00:00:00,5,5
4,1478,5,Nude 19,1/1/2002,40.0,28.0,100.0,70.0,0,5.jpg,0000-00-00 00:00:00,0000-00-00 00:00:00,2,1


In [30]:
gallery0 = (
    pd.DataFrame(artworks_dict)[["id", "title", "date_created", "xinch", "yinch", "xcm", "ycm", "filename", "category_id", "technique_id"]]
    .merge(pd.DataFrame(categories_dict)[["id", "name"]].rename(columns={"name": "category", "id": "category_id"}), on="category_id", how="left")
    .merge(pd.DataFrame(techniques_dict)[["id", "name"]].rename(columns={"name": "technique", "id": "technique_id"}), on="technique_id", how="left")
    .drop(columns=["category_id", "technique_id"])
)
gallery0.to_json("data/gallery.json", orient="records")
print(gallery0.shape)
gallery0.head(2)


(677, 10)


,id,title,date_created,xinch,yinch,xcm,ycm,filename,category,technique
0,1474,Victory,1/1/2002,24.0,16.0,60.0,40.0,1.jpg,Print,Engraving
1,1475,Dove 3,1/1/2002,16.0,28.0,40.0,70.0,2.jpg,Print,Engraving


In [29]:
gallery0.sort_values(by="filename")

,id,title,date_created,xinch,yinch,xcm,ycm,filename,category,technique
0,1474,Victory,1/1/2002,24.0,16.0,60.0,40.0,1.jpg,Print,Engraving
7,1481,Nude 1,1/1/2003,8.0,12.0,20.0,30.0,10.jpg,Drawing,Charcoal on paper
657,2131,Young Niusia,1/1/2012,18.0,24.0,45.0,60.0,1000.jpg,Portrait,Oil on canvas
658,2132,Old Niusia,1/1/2012,18.0,24.0,45.0,60.0,1002.jpg,Portrait,Oil on canvas
659,2133,Young Niusia,1/1/2012,18.0,24.0,45.0,60.0,1003.jpg,Portrait,Oil on canvas
...,...,...,...,...,...,...,...,...,...,...
69,1543,Dinner,1/1/2003,16.0,24.0,40.0,60.0,99.jpg,Poster,Engraving
653,2127,Young Niusia The Cat 5,1/1/2012,11.0,14.0,27.5,35.0,996.jpg,Portrait,Mixed technique on canvas
654,2128,Study of Women Hands - Cornelius Jonson,1/1/2011,8.0,12.0,20.0,30.0,997.jpg,Drawing,Charcoal on paper
655,2129,Lucian Freud,1/1/2011,18.0,24.0,45.0,60.0,998.jpg,Portrait,Mixed technique on canvas


In [27]:
images_for_studies = (
    pd.DataFrame(image_attachments_dict)
    [['study_id', 'image_id']]
    .groupby('study_id')
    ['image_id']
    .agg(list)
    .to_dict()
)
json.dump(images_for_studies, open("data/images_for_studies.json", "w"))
artworks_for_studies = (
    pd.DataFrame(attachments_dict)
    [['study_id', 'artwork_id']]
    .groupby('study_id')
    ['artwork_id']
    .agg(list)
    .to_dict()
)
json.dump(artworks_for_studies, open("data/artworks_for_studies.json", "w"))
artworks_filenames_and_titles = (
    gallery0
    [['id', 'filename', 'title']]
    .set_index('id')
    .to_dict(orient='index')
)
json.dump(artworks_filenames_and_titles, open("data/artworks_filenames_and_titles.json", "w"))
images_filenames = ( 
    pd.DataFrame(images_dict)
    [['id', 'filename']]
    .set_index('id')
    ['filename']
    .to_dict()
)
json.dump(images_filenames, open("data/images_filenames.json", "w"))

In [23]:
artworks_template = """
    <img src="gallery/{filename}" alt="{title}" />
"""
images_template = """
    <img src="images/{filename}" />
"""

body_template = """
    <h1>{name}</h1>
    {description}
    {artworks}
    {images}
"""

for _, row in pd.DataFrame(studies_dict).iterrows():
    id = row['id']
    slug = row['name'].lower().replace(" ", "-")
    name = row['name']
    description = row['description']
    images_data = images_for_studies.get(id, [])
    artworks_data = artworks_for_studies.get(id, [])

    images = "".join([images_template.format(filename=images_filenames[image_id]) for image_id in images_data])
    artworks = "".join([artworks_template.format(filename=artworks_filenames_and_titles[artwork_id]['filename'], title=artworks_filenames_and_titles[artwork_id]['title']) for artwork_id in artworks_data])

    body = body_template.format(name=name, description=description, artworks=artworks, images=images)

    with open(f"studies/{slug}.html", "w") as f:
        f.write(body)
